# Can we predict spam from email features?
The data looks like the UCI *Spambase* set: 57 numeric features (word / character frequencies and capital-letter run statistics) plus `Class` (1 = spam, 0 = not spam).

Plan: inspect the data, split before any modelling, compare a baseline with a few models using cross-validation, evaluate the chosen model once on a held-out test set, and look at which features matter.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay, precision_recall_curve,
                             accuracy_score, f1_score)
from sklearn.inspection import permutation_importance
SEED = 42
df = pd.read_csv('data.csv')
print(df.shape)
df.head()

## 1. Data quality

In [ ]:
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', df.duplicated().sum())
print(df['Class'].value_counts().rename({0: 'not spam', 1: 'spam'}))
print(df['Class'].value_counts(normalize=True).round(3).to_dict())

No missing values, and classes are moderately balanced (~39% spam). There are many exact duplicate rows. If left in, the same email could land in both train and test and inflate scores, so I drop them before splitting.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
X, y = df.drop(columns='Class'), df['Class']
print(X.shape, round(y.mean(), 3))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(len(X_train), len(X_test))

## 2. Exploration (training data only)

In [ ]:
tr = X_train.assign(Class=y_train)
m = tr.groupby('Class').mean().T
ratio = ((m[1] + 1e-3) / (m[0] + 1e-3)).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].barh(ratio.tail(10).index, ratio.tail(10), color='tab:red'); ax[0].set_title('Most spam-associated (mean spam / mean ham)')
ax[1].barh(ratio.head(10).index, ratio.head(10), color='tab:blue'); ax[1].set_title('Most ham-associated')
for a in ax: a.set_xscale('log')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, c in zip(ax, ['char_freq_!', 'char_freq_$', 'capital_run_length_longest']):
    for k, lab in [(0, 'not spam'), (1, 'spam')]:
        a.hist(np.log1p(tr.loc[tr.Class == k, c]), bins=40, alpha=.6, label=lab)
    a.set_title('log1p(' + c + ')'); a.legend()
plt.tight_layout(); plt.show()

Spam is associated with words like *free, remove, money, your, 000*, with `!`/`$` frequency and with long capital runs. Ham is associated with words like *george, hp, meeting, edu, re*; those reflect the mailbox the data came from, so a model may partly learn that mailbox's vocabulary. Features are heavily right-skewed, so I use a `log1p` transform for the linear model.

## 3. Models (5-fold stratified CV on the training set)

In [ ]:
log = FunctionTransformer(np.log1p)
models = {
    'Baseline (majority)': DummyClassifier(strategy='most_frequent'),
    'Logistic regression': make_pipeline(log, StandardScaler(), LogisticRegression(max_iter=2000)),
    'Random forest': RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
    'Gradient boosting': HistGradientBoostingClassifier(random_state=SEED),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = {}
for name, mdl in models.items():
    s = cross_validate(mdl, X_train, y_train, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])
    rows[name] = {k[5:]: v.mean() for k, v in s.items() if k.startswith('test_')}
cvres = pd.DataFrame(rows).T.round(3)
cvres

In [ ]:
best = cvres.drop('Baseline (majority)')['f1'].idxmax()   # selected on CV, not on test
print('Selected:', best)
final = models[best].fit(X_train, y_train)

## 4. Held-out test evaluation (used once)

In [ ]:
proba = final.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print(classification_report(y_test, pred, target_names=['not spam', 'spam'], digits=3))
print('ROC AUC:', round(roc_auc_score(y_test, proba), 4))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['not spam', 'spam']).plot(ax=ax[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[1]); plt.tight_layout(); plt.show()

In [ ]:
# Bootstrap 95% CIs on the test set
rng = np.random.default_rng(SEED); yt = y_test.to_numpy(); n = len(yt)
idx = [rng.integers(0, n, n) for _ in range(1000)]
bs = np.array([(accuracy_score(yt[i], pred[i]), f1_score(yt[i], pred[i])) for i in idx])
lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
print(f'accuracy 95% CI: [{lo[0]:.3f}, {hi[0]:.3f}]   F1 95% CI: [{lo[1]:.3f}, {hi[1]:.3f}]')

### Choosing a threshold
Flagging a real email as spam (false positive) is usually costlier than letting some spam through. The curve below shows the recall left if we demand very high precision.

In [ ]:
p, r, t = precision_recall_curve(y_test, proba)
plt.plot(r, p); plt.xlabel('recall'); plt.ylabel('precision'); plt.title('Precision-recall (test)'); plt.grid(alpha=.3); plt.show()
for target in (0.95, 0.98, 0.99):
    ok = np.where(p[:-1] >= target)[0]
    if len(ok):
        i = ok[0]; print(f'precision >= {target}: threshold {t[i]:.2f}, recall {r[i]:.3f}')

## 5. What drives the predictions?

In [ ]:
pi = permutation_importance(final, X_test, y_test, scoring='roc_auc', n_repeats=10, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values().tail(15)
imp.plot.barh(figsize=(7, 5), title='Permutation importance (drop in test AUC)'); plt.tight_layout(); plt.show()

## Conclusions
**Yes, spam is highly predictable from these features.** Cross-validated and held-out scores are well above the majority-class baseline (see the tables above for exact numbers).

Caveats:
- The data come from one mailbox and era (words like *george*, *hp*, *1999* are strong "ham" cues); the model would likely degrade on other users' email or modern spam.
- Features are word/character frequencies, so anything the dataset didn't track (images, links, headers) is invisible.
- Duplicates were removed to avoid optimistic estimates; the test set was used once, after model selection.
- Next steps: tune hyperparameters, pick the threshold from the real cost of false positives, and validate on fresh data.